# CS 839 HW1: Agent memory experiment (no-memory condition)

Runs the 12 tasks in `tasks.json`, **each in a fresh conversation**. Nothing carries over between tasks, so this is the *no memory* condition.

The loop is the same as in the course tool-calling exercise: user request → model tool calls → local Python results → model answer.

Run it from the repo root: it reads `data/` and imports from `utils/`.

The memory conditions go in the agent loop (section 4): add the memory to `INSTRUCTIONS`, and update it after each task from the returned trace.

## 1. Setup

When prompted, paste your team's API key. Never paste a key into a normal code cell or a Git repo.

In [ ]:
%pip install -q "openai>=1.0" pandas geopandas pyogrio pyproj shapely

In [1]:
import getpass
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

MODEL = os.getenv("CS839_MODEL", "gpt-5.6-luna")

if not os.getenv("OPENAI_API_KEY"):
    entered_key = getpass.getpass("Paste your team's API key (hidden): ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key

client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print(f"Model: {MODEL}")
print("API client ready." if client else "No key supplied. Local cells will still work.")

Model: gpt-5.6-luna
API client ready.


## 2. Tasks and tools

`utils/tools.py` holds the 8 tools and their descriptions (`TOOL_SPECS`). `data/tasks.json` holds the questions, answer formats, reference answers and known mistakes.

In [ ]:
from utils.tools import TOOLS, TOOL_SPECS
from utils.grader import grade, grade_canonical

SPEC = json.loads(Path("data/tasks.json").read_text())
TASKS = {t["id"]: t for t in SPEC["tasks"]}

pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"id": t["id"], "question": t["question"]} for t in SPEC["tasks"]])

Convert `TOOL_SPECS` to the Responses API format. Defaults are already stated in each description, so the JSON `default` keys are dropped.

In [ ]:
def strip_defaults(schema):
    if isinstance(schema, dict):
        return {k: strip_defaults(v) for k, v in schema.items() if k != "default"}
    if isinstance(schema, list):
        return [strip_defaults(v) for v in schema]
    return schema


TOOL_DEFS = [
    {
        "type": "function",
        "name": spec["name"],
        "description": spec["description"],
        "parameters": strip_defaults(spec["parameters"]),
        "strict": False,
    }
    for spec in TOOL_SPECS
]

MAX_TOOL_OUTPUT = 8000  # characters; list_regions alone is about 2,800


def execute_tool(name, arguments):
    if name not in TOOLS:
        return {"ok": False, "error": f"Unknown tool: {name}"}
    try:
        return {"ok": True, "result": TOOLS[name](**arguments)}
    except Exception as exc:  # bad arguments come back to the model as an error
        return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}


print([d["name"] for d in TOOL_DEFS])
print(json.dumps(execute_tool("get_population", {"names": ["Wisconsin", "Guam"]})))

## 3. Check the grader without spending API credit

Every reference answer must pass, and every known mistake must fail with its own label.

In [ ]:
for t in SPEC["tasks"]:
    assert grade_canonical(t, t["answer"])["correct"], t["id"]
    for fc in t["failure_conditions"]:
        g = grade_canonical(t, fc["answer"])
        assert not g["correct"] and g["label"] == fc["cause"], (t["id"], g)
print("Grader OK on all 12 reference answers and",
      sum(len(t["failure_conditions"]) for t in SPEC["tasks"]), "known mistakes.")

## 4. Agent loop: one task, fresh conversation

The instructions are the same for every task. The question and its answer format go in the user message.

In [ ]:
INSTRUCTIONS = (
    SPEC["scenario"] + " " + SPEC["system_prompt_definitions"] + " "
    "You cannot run code. Use the calculator tool for arithmetic and the rank tool for sorting."
)

MAX_STEPS = 15  # model calls per task; reaching it counts as a failure


def run_task(task, verbose=True):
    if client is None:
        raise RuntimeError("No API key is configured. Re-run the setup cell.")

    prompt = (f"{task['question']}\n\n"
              f"When you are done, reply with only a JSON object in this format:\n{task['answer_format']}")
    input_items = [{"role": "user", "content": prompt}]  # fresh conversation: no memory
    trace = {"task": task["id"], "question": task["question"], "calls": [], "final_answer": None,
             "model_calls": 0, "input_tokens": 0, "output_tokens": 0}
    if verbose:
        print(f"[{task['id']}] USER: {task['question']}\n")

    for step in range(1, MAX_STEPS + 1):
        response = client.responses.create(
            model=MODEL,
            instructions=INSTRUCTIONS,
            input=input_items,
            tools=TOOL_DEFS,
            tool_choice="auto",
        )
        trace["model_calls"] += 1
        if response.usage:
            trace["input_tokens"] += response.usage.input_tokens
            trace["output_tokens"] += response.usage.output_tokens

        input_items += response.output
        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            trace["final_answer"] = response.output_text
            if verbose:
                print(f"ASSISTANT: {trace['final_answer']}")
            break

        for call in function_calls:
            try:
                arguments = json.loads(call.arguments or "{}")
            except json.JSONDecodeError as exc:
                arguments, result = {}, {"ok": False, "error": f"Invalid JSON arguments: {exc}"}
            else:
                result = execute_tool(call.name, arguments)
            output = json.dumps(result)[:MAX_TOOL_OUTPUT]
            trace["calls"].append({"step": step, "name": call.name, "arguments": arguments, "result": output})
            if verbose:
                print(f"TOOL REQUEST: {call.name}({json.dumps(arguments)[:200]})")
                print(f"TOOL RESULT:  {output[:200]}{'...' if len(output) > 200 else ''}\n")
            input_items.append({"type": "function_call_output", "call_id": call.call_id, "output": output})
    else:
        trace["final_answer"] = None  # step limit reached

    trace["tool_calls"] = len(trace["calls"])
    trace.update(grade(task, trace["final_answer"]))
    if verbose:
        verdict = "CORRECT" if trace["correct"] else f"WRONG ({trace['label']})"
        print(f"\n{verdict} | tool calls {trace['tool_calls']} | tokens in {trace['input_tokens']}, "
              f"out {trace['output_tokens']}")
    return trace

## 5. Try one task first

In [ ]:
one = run_task(TASKS["T2"])

## 6. Run all 12 tasks

Each task is independent. Three runs per condition. Prices are for `gpt-5.6-luna`; the loop stops before $4.50.

In [ ]:
N_RUNS = 5
PRICE_PER_M_INPUT = 0.2   # USD per 1M input tokens for MODEL
PRICE_PER_M_OUTPUT = 1.2  # USD per 1M output tokens
BUDGET_USD = 4.50          # stop before going over


def cost(traces):
    if PRICE_PER_M_INPUT is None or PRICE_PER_M_OUTPUT is None:
        return None
    i = sum(t["input_tokens"] for t in traces)
    o = sum(t["output_tokens"] for t in traces)
    return i / 1e6 * PRICE_PER_M_INPUT + o / 1e6 * PRICE_PER_M_OUTPUT


Path("runs").mkdir(exist_ok=True)
all_traces = []
for run in range(1, N_RUNS + 1):
    for task in SPEC["tasks"]:
        spent = cost(all_traces)
        if spent is not None and spent > BUDGET_USD:
            raise RuntimeError(f"Budget reached: ${spent:.2f}")
        trace = run_task(task, verbose=False)
        trace["run"] = run
        all_traces.append(trace)
        print(f"run {run} {task['id']:>3}: {'correct' if trace['correct'] else 'WRONG  '}  "
              f"tools {trace['tool_calls']:>2}  {trace['label'] or ''}")

out_path = Path("runs") / f"no_memory_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(all_traces, indent=2, default=str))
print("\nSaved traces to", out_path)

## 7. Results

In [ ]:
df = pd.DataFrame([{
    "run": t["run"], "task": t["task"], "correct": t["correct"], "label": t["label"],
    "tool_calls": t["tool_calls"], "model_calls": t["model_calls"],
    "input_tokens": t["input_tokens"], "output_tokens": t["output_tokens"],
} for t in all_traces])

summary = df.groupby("task", sort=False).agg(
    success_rate=("correct", "mean"), tool_calls=("tool_calls", "mean"),
    input_tokens=("input_tokens", "mean"), output_tokens=("output_tokens", "mean"))
print(f"Accuracy: {df['correct'].mean():.0%}  |  tokens in {df['input_tokens'].sum():,}, "
      f"out {df['output_tokens'].sum():,}  |  cost: "
      + (f"${cost(all_traces):.3f}" if cost(all_traces) is not None else "set the prices above"))
summary

In [ ]:
# Wrong answers and their causes
df[~df["correct"]][["run", "task", "label"]]